In [1]:
import json
import re
import os

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
INPUT = os.path.join(BASE_DIR, "data", "raw", "dailydialog_converted.jsonl")
OUTPUT = os.path.join(BASE_DIR, "data", "raw", "dailydialog_cleaned.jsonl")

lines = open(INPUT, encoding="utf-8").readlines()
print(f"Loaded {len(lines)} raw lines")

Loaded 76051 raw lines


In [2]:
def fix_punct_spacing(text):
    # Remove space before . , ? ! : ; but keep the space after
    text = re.sub(r"\s+([.,?!:;])", r"\1", text)
    return text

sample = "You know that is tempting but is really not good for our fitness ."
print("Before:", sample)
print("After: ", fix_punct_spacing(sample))

Before: You know that is tempting but is really not good for our fitness .
After:  You know that is tempting but is really not good for our fitness.


In [3]:
for l in lines[:5]:
    obj = json.loads(l)
    print(fix_punct_spacing(obj["text"]))
    print("---")

### Instruction:
Say, Jim, how about going for a few beers after dinner?

### Response:
You know that is tempting but is really not good for our fitness.
---
### Instruction:
You know that is tempting but is really not good for our fitness.

### Response:
What do you mean? It will help us to relax.
---
### Context:
User: Say, Jim, how about going for a few beers after dinner?
Assistant: You know that is tempting but is really not good for our fitness.

### Instruction:
What do you mean? It will help us to relax.

### Response:
Do you really think so? I don't. It will just make us fat and act silly. Remember last time?
---
### Context:
User: You know that is tempting but is really not good for our fitness.
Assistant: What do you mean? It will help us to relax.

### Instruction:
Do you really think so? I don't. It will just make us fat and act silly. Remember last time?

### Response:
I guess you are right.But what shall we do? I don't feel like sitting at home.
---
### Context:
User: Wh

In [4]:
seen = set()
cleaned = []
dupes_removed = 0

for l in lines:
    l = l.strip()
    if not l:
        continue
    obj = json.loads(l)
    fixed_text = fix_punct_spacing(obj["text"])

    if fixed_text in seen:
        dupes_removed += 1
        continue

    seen.add(fixed_text)
    cleaned.append({"text": fixed_text})

print(f"Kept: {len(cleaned)}, Duplicates removed: {dupes_removed}")

Kept: 70308, Duplicates removed: 5743


In [5]:
# Re-check no empty responses survived, no double spaces introduced
empty_resp = sum(1 for c in cleaned if c["text"].rstrip().endswith("### Response:"))
double_space = sum(1 for c in cleaned if "  " in c["text"])

print(f"Empty responses: {empty_resp}")
print(f"Rows with double spaces (should be near 0): {double_space}")

# Print 5 random cleaned samples to eyeball
import random
for c in random.sample(cleaned, 5):
    print(c["text"])
    print("=" * 40)

Empty responses: 0
Rows with double spaces (should be near 0): 0
### Instruction:
You are late.

### Response:
I'm sorry, it was too cold, and my car couldn't start. I had one to the garage with the heater. I tried to call you, but you couldn't get connection.
### Instruction:
Can I help you, madam?

### Response:
Yes, I'm looking for a cap. Size 16.
### Instruction:
Have you heard about the robbery?

### Response:
I saw the whole thing happen.
### Context:
User: The heating controls don't work anymore, so it always feels like it's about 100 degrees in the car — even in the summer!
Assistant: Anything else?

### Instruction:
The brakes don't really work that well anymore either.

### Response:
Why don't you get it all fixed?
### Context:
User: Mon and I got in another fight, Boris.
Assistant: oh, Iris! what was it about this time?

### Instruction:
it was over food. I simply wanted some fried chicken but she said no.

### Response:
I believe she was right. You must know that fried food

In [6]:
with open(OUTPUT, "w", encoding="utf-8") as out:
    for c in cleaned:
        out.write(json.dumps(c) + "\n")

print(f"Saved {len(cleaned)} cleaned examples to {OUTPUT}")

Saved 70308 cleaned examples to C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\dailydialog_cleaned.jsonl


In [7]:
import json
import re
import os

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
PATH = os.path.join(BASE_DIR, "data", "raw", "dailydialog_cleaned.jsonl")

def fix_sentence_boundaries(text):
    # Add space after '.' when directly followed by a capital letter (mid-sentence run-on)
    text = re.sub(r'([a-z])\.([A-Z])', r'\1. \2', text)
    return text

def fix_curly_apostrophes(text):
    # Normalize "word ’ s" -> "word's" (collapse spaces around curly apostrophe, then convert to straight)
    text = re.sub(r"\s+’\s+", "'", text)
    text = text.replace("’", "'")  # catch any remaining unspaced curly quotes
    return text

lines = open(PATH, encoding="utf-8").readlines()
print(f"Loaded {len(lines)} lines")

fixed = []
for l in lines:
    l = l.strip()
    if not l:
        continue
    obj = json.loads(l)
    text = obj["text"]
    text = fix_sentence_boundaries(text)
    text = fix_curly_apostrophes(text)
    fixed.append({"text": text})

print(f"Fixed {len(fixed)} rows")

# Eyeball a few before/after on rows we know had the issues
import random
sample_idx = random.sample(range(len(lines)), 5)
for i in sample_idx:
    before = json.loads(lines[i])["text"]
    after = fixed[i]["text"]
    if before != after:
        print("BEFORE:", before[:150])
        print("AFTER: ", after[:150])
        print("=" * 40)

Loaded 70308 lines
Fixed 70308 rows
BEFORE: ### Context:
User: Roger? I need figures for accounting. Have finished the calculations?
Assistant: I ’ m just finishing now, ma ’ am. I ’ ll have the
AFTER:  ### Context:
User: Roger? I need figures for accounting. Have finished the calculations?
Assistant: I'm just finishing now, ma'am. I'll have them read
BEFORE: ### Instruction:
This website offers very convenient air tickets booking service. It is quick and accurate.

### Response:
Yes, I once booked there. T
AFTER:  ### Instruction:
This website offers very convenient air tickets booking service. It is quick and accurate.

### Response:
Yes, I once booked there. T


In [8]:
# Re-check both target patterns are gone
no_space_after = sum(1 for f in fixed if re.search(r"[a-z]\.[A-Z]", f["text"]))
curly_left = sum(1 for f in fixed if "’" in f["text"])
double_space = sum(1 for f in fixed if "  " in f["text"])

print(f"Remaining run-ons: {no_space_after}")
print(f"Remaining curly apostrophes: {curly_left}")
print(f"Double spaces introduced: {double_space}")

if no_space_after == 0 and curly_left == 0:
    with open(PATH, "w", encoding="utf-8") as out:
        for f in fixed:
            out.write(json.dumps(f) + "\n")
    print(f"Overwritten: {PATH}")
else:
    print("Not saved — check remaining issues above before overwriting.")

Remaining run-ons: 0
Remaining curly apostrophes: 0
Double spaces introduced: 0
Overwritten: C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\dailydialog_cleaned.jsonl


In [9]:
import json

BASE_DIR = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational"
PATH = BASE_DIR + r"\data\raw\dailydialog_cleaned.jsonl"

lines = open(PATH, encoding="utf-8").readlines()

seen = set()
final = []
dupes_removed = 0

for l in lines:
    l = l.strip()
    if not l:
        continue
    obj = json.loads(l)
    if obj["text"] in seen:
        dupes_removed += 1
        continue
    seen.add(obj["text"])
    final.append(obj)

print(f"Kept: {len(final)}, Duplicates removed: {dupes_removed}")

with open(PATH, "w", encoding="utf-8") as out:
    for f in final:
        out.write(json.dumps(f) + "\n")

print(f"Final saved: {PATH}")

Kept: 69569, Duplicates removed: 739
Final saved: C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\dailydialog_cleaned.jsonl


In [11]:
import json, random

STAGE_A_PATH = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\train.jsonl"   # your Stage A file
STAGE_B_PATH = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\raw\dailydialog_cleaned.jsonl"
OUTPUT = r"C:\Users\MIND & MATTER\Desktop\ScratchLLM-95_conversational\data\processed\stage_b_train.jsonl"

stage_a = [json.loads(l) for l in open(STAGE_A_PATH, encoding="utf-8")]
stage_b = [json.loads(l) for l in open(STAGE_B_PATH, encoding="utf-8")]

replay_ratio = 0.15  # 15% replay
replay_n = int(len(stage_b) * replay_ratio / (1 - replay_ratio))
replay_sample = random.sample(stage_a, min(replay_n, len(stage_a)))

combined = stage_b + replay_sample
random.shuffle(combined)

with open(OUTPUT, "w", encoding="utf-8") as out:
    for row in combined:
        out.write(json.dumps(row) + "\n")

print(f"Stage B: {len(stage_b)}, Replay: {len(replay_sample)}, Total: {len(combined)}")

Stage B: 69569, Replay: 12276, Total: 81845
